# NGS Data Organization and Storage Strategy

This notebook demonstrates how to design and maintain a consistent folder hierarchy for Next-Generation Sequencing (NGS) data processing. 

We'll cover:
- Folder setup and naming conventions
- Storage recommendations within the class and personal use within the HPC cluster
- Example automation with bash (also included in [Module_2/Resources/DemoScripts](../resources/demo_scripts/README.md))

**NOTE: The grade for this exercise, M2E1, is based on the completion of creating the `m3_project_pseudo_alignment` folder and it's dependencies.**


```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## Folder Organization - Proper organization makes life simple

Within any computational environment, proper organization of data is critical. Using consistent naming, keeping proper metadata sheets, and using well-thought-out directory trees will allow for easy processing. When we start a project or a pipeline, it is always good to create a "root" folder. This will be the central place where all files are kept and prevents the mixing of outputs when you are working with many similarly named files. 

#### Inside the root folder, we will generally have several consistent paths:
- `data` folder: Where the raw reads and outputs are stored.
- `references` folder: Records that are needed for input to programs outside of the raw / output data. This can include reference genomes, `TX2GENE` tables, or other materials needed for our pipelines.
- `results` folder: Here, the outputs of tools and programs we use within our pipelines are stored.
- `scripts` folder: Storage for all of the scripts that are utilized within the pipeline. Can include bash, R, Python, or other scripts
- `Enviornments` or `images` folder (Optional): If you are working with a complicated pipeline that you would like to make reproducible across machines, you can build enviornments locally with an environment manager such as `conda`, or use a container manager like [`docker`](https://www.docker.com/) or [`Apptainer` (previously `Singularity`)](https://apptainer.org/). These differ from conda in the sense that the containers function as enviornments that are completely portable, meaning their dependencies can be moved across systems for reproducibility. Pipelines like [`nf-core`](https://nf-co.re/) are tailored to these container enviornments to create reproducible pipelines using [Next-Flow](https://www.nextflow.io/) for processing NGS data.

**Below is an example of a basic project root folder that could be implemented in an RNA-Seq pipeline**

```
project_root/
├── data/
│ ├── raw/ → original FASTQ files from sequencing
│ ├── trimmed/ → output after adapter/quality trimming
│ ├── aligned/ → alignment results (SAM/BAM files) as finalized tracks for genome browsers
│ └── counts/ → processed read count tables and matrices(gene × sample)
│
├── references/
│ ├── genome/ → reference genome FASTA (.fa/.fasta)
│ ├── gtf/ → annotation files (.gtf/.gff)
│ ├── indices/ → prebuilt indices for tools (STAR, HISAT2, etc.)
│ └── other/ → optional: motifs, BED files, control datasets
│
├── results/
│ ├── qc/ → Fastp, MultiQC, or trimming logs
│ ├── aligned/ → alignment results (SAM/BAM files)
│ ├── differential_expression/ → normalized counts (TPM/FKPM values for samples)
│ ├── visualization/ → PCA plots, heatmaps, volcano plots
│ └── reports/ → notebooks, summary PDFs
│
└── scripts/
├── preprocessing/
├── alignment/
├── analysis/
└── notebooks/
```


# Storage management within the course and within Hellbender's HPC storage

In the class folders, we will have a `Project` folder for each week we are processing NGS data. This folder will be contained directly within the `Modules` folder and titled based on the weekly project. This week's project folder is called `m2_project_fastp_usage`, and within it you will find the standard `data`, `results`, `references`, and `scripts` folders. Within this exercise, you will set up your `Module_3` project folder, called `m3_project_pseudo_alignment` for RNA-Seq using pseudo-alignment. Please follow along with the video below to create these folders and learn about file storage within the HPC cluster. 

**NOTE: The grade for this exercise, M2E1, is based on the completion of creating this folder.**

### M2E1V8 - Prudent File Storage Organization:

<iframe width="560" height="315" src="https://www.youtube.com/embed/wDYgl-L4lUg?si=-zJOm8sYOQvdv5xr" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

Within the HPC cluster, if you are creating a personal project folder (ie, not going to be stored within a shared lab folder) it should be placed within the `data` symlink within your `home` folder. This `data` folder is your personal storage on Hellbender and contains 500 GB of space to use with each researcher account. Your `home` folder only contains 50 GB of storage, and should be used for files that need to be accessed easily or do not consume much space. Below are some notes on the storage system within our HPC Cluster: 
- It is recommended that you change your conda environment storage to your `data` folder by modifying the `.condarc` file within your home directory. This can be seen in `~` if you use the command `ls -a`. Make sure to read over the [Hellbender Documentation](https://docs.itrss.umsystem.edu/pub/hpc/hellbender) before modifying this, as it can create dependency issues within your current enviornments if it is moved improperly. Contact ITRSS with any issues regarding this.
- If you would like to transfer files between the HPC server and your personal computer, there are several ways to do this:
    - Use [Globus](https://www.globus.org/) file transfer service to move files between directories with a GUI
    - Use the [`scp`](https://man7.org/linux/man-pages/man1/scp.1.html) command within a terminal to transfer files between your personal computer and the HPC server
    - Upload files to your home directory within Jupyter Notebook (Not recommended, as the file size limit is quite small)
- If your lab has a storage directory within the `RDE` or VAST server, then you are able to store project files there. Each lab should be allocated 10 TB of storage for free if requested. Contact ITRSS or refer to the [Hellbender Documentation](https://docs.itrss.umsystem.edu/pub/hpc/hellbender) for more information. 

## Scripting for Creation and Use of Directories

**When writing scripts within bash, it is essential that you declare the names of your directories as variables within the shell**; this allows for these scripts to be modified efficiently. When writing scripts, such as the example below, remember that you may change file names or paths later on, so declaring variables makes code dynamic and usable across pipelines. 

Below is an example of a script that creates a project folder containing directories standard `data`, `results`, `references`, and `scripts` within the current working directory.

In [ ]:
%%bash
# ==============================================================
# Script Name: create_ngs_project_structure.sh
# Description: Sets up standardized folder structure for NGS projects
# Author: Pierce Taylor
# ==============================================================

# Exit if any command fails
set -e

# ---[ 1. Get project name or use default ]---
PROJECT_NAME=${1:-NGS_Project}
BASE_DIR=${2:-"$HOME/projects"}        # default base path under ~/projects
PROJECT_ROOT="$BASE_DIR/$PROJECT_NAME"

echo "Creating project structure at: $PROJECT_ROOT"
echo "----------------------------------------------------"

# ---[ 2. Define subfolders ]---
declare -a DIRS=(
    # Data folders
    "data/raw"
    "data/trimmed"
    "data/aligned"
    "data/counts"

    # References
    "references/genome"
    "references/indices"
    "references/other"

    # Results
    "results/qc"
    "results/differential_expression"
    "results/visualization"
    "results/reports"

    # Scripts
    "scripts/preprocessing"
    "scripts/alignment"
    "scripts/analysis"
    "scripts/notebooks"
)

# ---[ 3. Create the directory structure ]---
for dir in "${DIRS[@]}"; do
    mkdir -p "$PROJECT_ROOT/$dir"
    echo "Created: $PROJECT_ROOT/$dir"
done

# ---[ 4. Create base README.md ]---
cat <<EOF > "$PROJECT_ROOT/README.md"
# Project: $PROJECT_NAME
Created on: $(date +"%Y-%m-%d")

## Folder Overview
- **data/**: Raw and processed NGS reads (FASTQ, BAM, count tables)
- **references/**: Genome, annotation (GTF/GFF), and indices
- **results/**: QC, differential expression, and visualizations
- **scripts/**: All preprocessing, alignment, and analysis scripts

## Notes
Keep raw FASTQ files immutable.
Store analysis environment files here (e.g. environment.yml).
EOF

echo "Created README.md"
